In [76]:
# 1
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, desc, count
from functools import reduce
import pandas as pd

spark = SparkSession.builder \
    .appName("Pagila Analysis") \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

In [77]:
#  2
film = spark.read.option("header", True).csv("data/film.csv")
actor = spark.read.option("header", True).csv("data/actor.csv")
category = spark.read.option("header", True).csv("data/category.csv")
film_category = spark.read.option("header", True).csv("data/film_category.csv")
inventory = spark.read.option("header", True).csv("data/inventory.csv")
customer = spark.read.option("header", True).csv("data/customer.csv")
address = spark.read.option("header", True).csv("data/address.csv")
city = spark.read.option("header", True).csv("data/city.csv")
rental = spark.read.option("header", True).csv("data/rental.csv")

# Объединение payment из 7 файлов
payments = []
for i in range(1, 8):
    p = spark.read.option("header", True).csv(f"data/payment_p2022_0{i}.csv")
    payments.append(p)
payment = reduce(lambda a, b: a.union(b), payments)


In [78]:
#3  Количество фильмов по категориям
cat_count = film_category.join(category, "category_id").groupBy("name").count().orderBy(desc("count"))


cat_count_pd = cat_count.toPandas()
cat_count_pd


,name,count
0,Drama,152
1,Music,152
2,Travel,151
3,Foreign,150
4,Games,150
5,Children,150
6,Action,149
7,Sci-Fi,149
8,Animation,148
9,Family,147


In [79]:
# 4: 10 актёров с наибольшим количеством аренд (без payment)

film_actor = spark.read.option("header", True).csv("data/film_actor.csv")

actor_rentals = film_actor.join(inventory, "film_id").join(rental, "inventory_id")

actor_rent_count = actor_rentals.groupBy("actor_id").agg(count("rental_id").alias("rentals")) \
    .join(actor, "actor_id").select("first_name", "last_name", "rentals").orderBy(desc("rentals"))

actor_rent_count_pd = actor_rent_count.limit(10).toPandas()
actor_rent_count_pd



,first_name,last_name,rentals
0,GINA,DEGENERES,753
1,MATTHEW,CARREY,678
2,MARY,KEITEL,674
3,ANGELA,WITHERSPOON,654
4,WALTER,TORN,640
5,HENRY,BERRY,612
6,JAYNE,NOLTE,611
7,VAL,BOLGER,605
8,SANDRA,KILMER,604
9,SEAN,GUINESS,599


In [80]:
# 5: категория с наибольшей суммой платежей

#  соединяем inventory + rental + payment
rental_payment = inventory.join(rental, "inventory_id").join(payment, "rental_id")

# film + film_category + category
film_cat = film.join(film_category, "film_id").join(category, "category_id")

# Соединяем film_cat с rental_payment через film_id → inventory → rental → payment
cat_payment = film_cat.join(rental_payment, "film_id") \
    .groupBy("name") \
    .agg(spark_sum("amount").alias("total_amount")) \
    .orderBy(desc("total_amount"))

# категория с наибольшей суммой
cat_payment_pd = cat_payment.limit(1).toPandas()
cat_payment_pd



,name,total_amount
0,Foreign,10507.67


In [81]:
# 6: фильмы, которых нет в inventory
film_not_in_inventory = film.join(inventory, "film_id", "left_anti")
film_not_in_inventory_pd = film_not_in_inventory.limit(10).toPandas()
film_not_in_inventory_pd


,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,last_update,special_features,fulltext
0,14,ALICE FANTASIA,A Emotional Drama of a A Shark And a Database ...,2013,4,None,6,0.99,94,23.99,NC-17,2022-09-10 18:46:03.905795+02,"""{Trailers,""""Deleted Scenes""""","""""Behind the Scenes""""}"""
1,33,APOLLO TEEN,A Action-Packed Reflection of a Crocodile And ...,2013,6,None,5,2.99,153,15.99,PG-13,2022-09-10 18:46:03.905795+02,"""{Trailers,Commentaries,""""Deleted Scenes""""","""""Behind the Scenes""""}"""
2,36,ARGONAUTS TOWN,A Emotional Epistle of a Forensic Psychologist...,2011,3,None,7,0.99,127,12.99,PG-13,2022-09-10 18:46:03.905795+02,"{Trailers,Commentaries}",'abandon':20 'argonaut':1 'butler':12 'challen...
3,38,ARK RIDGEMONT,A Beautiful Yarn of a Pioneer And a Monkey who...,2009,1,None,6,0.99,68,25.99,NC-17,2022-09-10 18:46:03.905795+02,"""{Trailers,Commentaries,""""Deleted Scenes""""","""""Behind the Scenes""""}"""
4,41,ARSENIC INDEPENDENCE,A Fanciful Documentary of a Mad Cow And a Woma...,2006,2,None,4,0.99,137,17.99,PG,2022-09-10 18:46:03.905795+02,"""{Trailers,""""Deleted Scenes""""","""""Behind the Scenes""""}"""
5,87,BOONDOCK BALLROOM,A Fateful Panorama of a Crocodile And a Boy wh...,2017,1,None,7,0.99,76,14.99,NC-17,2022-09-10 18:46:03.905795+02,"""{""""Behind the Scenes""""}""",'ballroom':2 'boondock':1 'boy':11 'crocodil':...
6,108,BUTCH PANTHER,A Lacklusture Yarn of a Feminist And a Databas...,2013,6,None,6,0.99,67,19.99,PG-13,2022-09-10 18:46:03.905795+02,"""{Trailers,Commentaries,""""Deleted Scenes""""}""",'administr':12 'butch':1 'databas':11 'face':1...
7,128,CATCH AMISTAD,A Boring Reflection of a Lumberjack And a Femi...,2007,3,None,7,0.99,183,10.99,G,2022-09-10 18:46:03.905795+02,"""{Trailers,""""Behind the Scenes""""}""",'amistad':2 'bore':4 'catch':1 'discov':14 'fe...
8,144,CHINATOWN GLADIATOR,A Brilliant Panorama of a Technical Writer And...,2013,4,None,7,4.99,61,24.99,PG,2022-09-10 18:46:03.905795+02,"""{Trailers,Commentaries,""""Deleted Scenes""""}""",'ancient':19 'brilliant':4 'butler':17 'chinat...
9,148,CHOCOLATE DUCK,A Unbelieveable Story of a Mad Scientist And a...,2009,1,None,3,2.99,132,13.99,R,2022-09-10 18:46:03.905795+02,"""{Trailers,Commentaries,""""Behind the Scenes""""}""",'ancient':20 'china':21 'chocol':1 'compos':18...


In [82]:
# 7: топ 3 актеров для категории Children

#  Выбираем категорию Children
children_cat = category.filter(col("name") == "Children")

# Фильмы категории Children
children_films = film_category.join(children_cat, "category_id")

# Соединяем с таблицей film_actor для получения актеров
film_actor = spark.read.option("header", True).csv("data/film_actor.csv")
children_actors = children_films.join(film_actor, "film_id") \
                                .join(actor, "actor_id")

# Считаем количество фильмов на актера
actor_count_children = children_actors.groupBy("first_name", "last_name") \
                                      .agg(count("film_id").alias("film_count")) \
                                      .orderBy(desc("film_count"))

#  Выбираем топ 3 актеров 
top_count = actor_count_children.limit(3).collect()[-1]["film_count"]
top_actors_children = actor_count_children.filter(col("film_count") >= top_count)

#  результат
top_actors_children_pd = top_actors_children.toPandas()
top_actors_children_pd


,first_name,last_name,film_count
0,SIDNEY,CROWE,9
1,RICHARD,PENN,9
2,EWAN,GOODING,9


In [83]:
#  8: активные и неактивные клиенты по городам. Sort by the number of inactive customers in descending order. 
from pyspark.sql.functions import when

customer_city = customer.join(address, "address_id").join(city, "city_id")

# получили таблицу, где у каждой строки есть имя города и статус клиента ( 1 или 0).

# сначала считаем активных, потом - неактивных
# если активен - ставим 1, иначе - 0, затем считаем сумму. Тоже самое для неактивных.  
city_status = customer_city.groupBy("city").agg(
        spark_sum(when(col("active") == 1, 1).otherwise(0)).alias("active_customers"),
        spark_sum(when(col("active") == 0, 1).otherwise(0)).alias("inactive_customers")
    ).orderBy(desc("inactive_customers"))

city_status_pd = city_status.toPandas()
city_status_pd


,city,active_customers,inactive_customers
0,Uluberia,0,1
1,Najafabad,0,1
2,Pingxiang,0,1
3,Xiangfan,0,1
4,Kumbakonam,0,1
...,...,...,...
592,San Juan Bautista Tuxtepec,1,0
593,Jelets,1,0
594,Brescia,1,0
595,Teboksary,1,0


In [84]:
# 9: категории фильмов с наибольшими часами аренды для городов на "A"
rental_hours = rental.withColumn("rental_hours", (col("return_date").cast("timestamp").cast("long") - col("rental_date").cast("timestamp").cast("long"))/3600)

city_a = city.filter(col("city").startswith("A"))
city_a_rental = rental_hours.join(inventory, "inventory_id") \
    .join(film_category, "film_id") \
    .join(category, "category_id") \
    .join(customer.join(address, "address_id").join(city, "city_id"), "customer_id") \
    .filter(col("city").startswith("A")) \
    .groupBy("name") \
    .agg(spark_sum("rental_hours").alias("total_hours")) \
    .orderBy(desc("total_hours"))

city_a_rental_pd = city_a_rental.limit(1).toPandas()
city_a_rental_pd



,name,total_hours
0,Children,24428.0


In [85]:
#  для городов с "-"
city_dash = city.filter(col("city").contains("-"))
city_dash_rental = rental_hours.join(inventory, "inventory_id") \
    .join(film_category, "film_id") \
    .join(category, "category_id") \
    .join(customer.join(address, "address_id").join(city, "city_id"), "customer_id") \
    .filter(col("city").contains("-")) \
    .groupBy("name") \
    .agg(spark_sum("rental_hours").alias("total_hours")) \
    .orderBy(desc("total_hours"))

city_dash_rental_pd = city_dash_rental.limit(1).toPandas()
city_dash_rental_pd

,name,total_hours
0,Drama,14556.033333
